# AMPI Demo — Approximate Nearest Neighbors on MNIST

**Dataset**: MNIST (70 000 images, d = 784)

**Methods compared**:

| Method | Type | Library |
|---|---|---|
| FAISS FlatL2 | Exact brute force | FAISS |
| PyKeOps | Exact brute force (lazy) | PyKeOps |
| FAISS IVF | Approximate (inverted file) | FAISS |
| AMPI Binary | Approximate (sorted projections) | this repo |
| AMPI Hashing | Approximate (p-stable LSH) | this repo |

We measure **Recall\@k** (fraction of true top-k neighbours returned) and **ms/query**.

See `README.md` for the mathematical background.

In [ ]:
import sys, os, time, struct
import numpy as np
import faiss

sys.path.insert(0, 'ampi')
from ampi import AMPIBinaryIndex as AMPIBinary
from ampi import AMPIHashIndex   as AMPIHashing

try:
    import torch
    from pykeops.torch import LazyTensor
    HAS_KEOPS = True
    print('PyKeOps available')
except ImportError:
    HAS_KEOPS = False
    print('PyKeOps not installed — skipping (pip install pykeops torch)')

SEED = 42
K    = 10      # neighbours
NQ   = 500     # number of query points
print(f'faiss version: {faiss.__version__}')

In [ ]:
# ── Load MNIST from the raw IDX files (no torchvision required) ──────────────

def _read_idx_images(path):
    with open(path, 'rb') as f:
        magic, n, rows, cols = struct.unpack('>IIII', f.read(16))
        assert magic == 2051
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(n, rows * cols)

def _read_idx_labels(path):
    with open(path, 'rb') as f:
        magic, n = struct.unpack('>II', f.read(8))
        assert magic == 2049
        return np.frombuffer(f.read(), dtype=np.uint8)

root = 'data/MNIST/raw'
X_train = _read_idx_images(f'{root}/train-images-idx3-ubyte')
X_test  = _read_idx_images(f'{root}/t10k-images-idx3-ubyte')
y_train = _read_idx_labels(f'{root}/train-labels-idx1-ubyte')
y_test  = _read_idx_labels(f'{root}/t10k-labels-idx1-ubyte')

# Stack and normalise to [0, 1] float32
X = np.vstack([X_train, X_test]).astype(np.float32) / 255.0
y = np.concatenate([y_train, y_test])
n, d = X.shape

rng     = np.random.default_rng(SEED)
q_idx   = rng.choice(n, NQ, replace=False)
queries = X[q_idx]   # shape (NQ, 784)

print(f'Dataset: n={n:,}  d={d}  queries={NQ}')

In [ ]:
# ── Ground truth: FAISS FlatL2 (exact brute force) ───────────────────────────

flat = faiss.IndexFlatL2(d)
flat.add(X)

t0 = time.perf_counter()
_, I_true = flat.search(queries, K)
flat_total_ms = (time.perf_counter() - t0) * 1000

print(f'FAISS FlatL2  total {flat_total_ms:.0f} ms  ({flat_total_ms/NQ:.2f} ms/query)')

In [ ]:
# ── PyKeOps brute force (lazy, no O(n²) materialisation) ─────────────────────

if HAS_KEOPS:
    X_t  = torch.from_numpy(X)
    Q_t  = torch.from_numpy(queries)
    X_i  = LazyTensor(Q_t[:, None, :])   # (NQ, 1, d)
    X_j  = LazyTensor(X_t[None, :, :])   # (1, n, d)
    D_ij = ((X_i - X_j) ** 2).sum(-1)   # (NQ, n)

    t0 = time.perf_counter()
    I_keops = D_ij.argKmin(K, dim=1).numpy()   # (NQ, K)
    keops_total_ms = (time.perf_counter() - t0) * 1000

    # Verify it matches FAISS
    overlap = np.mean([
        len(set(I_true[i]) & set(I_keops[i])) / K
        for i in range(NQ)
    ])
    print(f'PyKeOps       total {keops_total_ms:.0f} ms  ({keops_total_ms/NQ:.2f} ms/query)  '
          f'recall@{K}={overlap:.4f} (vs FAISS Flat)')
else:
    print('PyKeOps skipped.')

In [ ]:
# ── Build AMPI and FAISS IVF indices ─────────────────────────────────────────
#
# MNIST: d=784, r_nn is small for nearest same-class images.
# σ_proj = r_nn/√d — empirically r_nn≈4-8, so σ_proj≈0.14-0.29
# → bucket_size ≈ 0.2-0.5 is appropriate here (unlike random Gaussian data).
# The sweep in the last section will confirm this.

print('Building AMPI Binary  (L=16)…', end=' ', flush=True)
t0 = time.perf_counter()
idx_bin = AMPIBinary(X, num_projections=16, seed=SEED)
print(f'{time.perf_counter()-t0:.2f}s')

print('Building AMPI Hashing (L=16, w=0.3)…', end=' ', flush=True)
t0 = time.perf_counter()
idx_hash = AMPIHashing(X, num_projections=16, bucket_size=0.3, seed=SEED)
print(f'{time.perf_counter()-t0:.2f}s')

nlist = max(64, int(n * 0.01))
print(f'Building FAISS IVF    (nlist={nlist})…', end=' ', flush=True)
t0 = time.perf_counter()
quant = faiss.IndexFlatL2(d)
ivf   = faiss.IndexIVFFlat(quant, d, nlist, faiss.METRIC_L2)
ivf.train(X); ivf.add(X)
print(f'{time.perf_counter()-t0:.2f}s')

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def recall_at_k(I_true, I_approx, k):
    hits = sum(len(set(t[:k]) & set(a[:k])) for t, a in zip(I_true, I_approx))
    return hits / (len(I_true) * k)

def warmup_time(fn, items, warmup=5):
    for x in items[:warmup]: fn(x)
    t0 = time.perf_counter()
    for x in items:          fn(x)
    return (time.perf_counter() - t0) / len(items) * 1000

def avg_cands(idx, queries, **kw):
    return int(np.mean([len(idx.query_candidates(q, **kw)) for q in queries[:20]]))

def run_config(label, query_fn, cands_fn=None):
    rows = [query_fn(q) for q in queries]
    I    = np.array([np.pad(r, (0, max(0, K - len(r))), constant_values=-1) for r in rows])
    ms   = warmup_time(query_fn, queries)
    cstr = f'{cands_fn():>8,}' if cands_fn else f'{"—":>8}'
    r1   = recall_at_k(I_true, I, 1)
    r5   = recall_at_k(I_true, I, 5)
    rk   = recall_at_k(I_true, I, K)
    print(f'  {label:<26}  {ms:>8.2f}  {cstr}  {r1:>6.3f}  {r5:>6.3f}  {rk:>6.3f}')

In [ ]:
# ── Summary comparison ────────────────────────────────────────────────────────

print(f'  {"Method":<26}  {"ms/q":>8}  {"cands":>8}  {"R@1":>6}  {"R@5":>6}  {f"R@{K}":>6}')
print('  ' + '-' * 72)

# FAISS Flat (exact)
ms = warmup_time(lambda q: flat.search(q[None], K), queries)
r1 = recall_at_k(I_true, I_true, 1)   # always 1.0
print(f'  {"Flat L2 (exact)":<26}  {ms:>8.2f}  {n:>8,}  {r1:>6.3f}  {1.0:>6.3f}  {1.0:>6.3f}')

# PyKeOps
if HAS_KEOPS:
    ms_keops = keops_total_ms / NQ
    print(f'  {"PyKeOps (exact)":<26}  {ms_keops:>8.2f}  {n:>8,}  {1.0:>6.3f}  {1.0:>6.3f}  {1.0:>6.3f}')

# FAISS IVF
for nprobe in [1, 5, 20]:
    ivf.nprobe = nprobe
    rows = [ivf.search(q[None], K)[1][0] for q in queries]
    I_ivf = np.array(rows)
    ms  = warmup_time(lambda q: ivf.search(q[None], K), queries)
    r1  = recall_at_k(I_true, I_ivf, 1)
    r5  = recall_at_k(I_true, I_ivf, 5)
    rk  = recall_at_k(I_true, I_ivf, K)
    cds = nprobe * (n // nlist)
    print(f'  {f"IVF nprobe={nprobe}":<26}  {ms:>8.2f}  {cds:>8,}  {r1:>6.3f}  {r5:>6.3f}  {rk:>6.3f}')

# AMPI Binary
for w in [25, 100, 400]:
    run_config(
        f'AMPI Binary  w={w}',
        lambda q, w=w: idx_bin.query(q, k=K, window_size=w)[2],
        lambda w=w: avg_cands(idx_bin, queries, window_size=w),
    )

# AMPI Hashing
for p in [1, 2, 4]:
    run_config(
        f'AMPI Hash    w=0.3 p={p}',
        lambda q, p=p: idx_hash.query(q, k=K, probes=p)[2],
        lambda p=p: avg_cands(idx_hash, queries, probes=p),
    )

## Hashing degradation: `bucket_size` sweep

The key calibration: `bucket_size` must be roughly `r_nn / √d`.

For MNIST (d=784), σ_proj is smaller than for random Gaussian data — nearest images differ mainly in a few pixels, so their 1-D projection difference is small.  
We sweep `bucket_size` from very small to large (with `probes=3` fixed) to reveal the sweet spot.

In [ ]:
print(f'  AMPI Hashing (L=16, probes=3 fixed) — vary bucket_size')
print(f'  {"bucket_size":>12}  {"±window":>8}  {"cands":>8}  {"ms/q":>8}  {"R@1":>6}  {"R@5":>6}  {f"R@{K}":>6}')
print('  ' + '-' * 72)

for bsz in [0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0, 2.0]:
    h = AMPIHashing(X, num_projections=16, bucket_size=bsz, seed=SEED)
    rows = [h.query(q, k=K, probes=3)[2] for q in queries]
    I    = np.array([np.pad(r, (0, max(0, K - len(r))), constant_values=-1) for r in rows])
    ms   = warmup_time(lambda q, h=h: h.query(q, k=K, probes=3), queries)
    cds  = int(np.mean([len(h.query_candidates(q, probes=3)) for q in queries[:20]]))
    r1   = recall_at_k(I_true, I, 1)
    r5   = recall_at_k(I_true, I, 5)
    rk   = recall_at_k(I_true, I, K)
    win  = f'±{2*bsz:.2f}'
    print(f'  {bsz:>12.3f}  {win:>8}  {cds:>8,}  {ms:>8.2f}  {r1:>6.3f}  {r5:>6.3f}  {rk:>6.3f}')

## Binary vs Hashing: iso-recall curve

For a fixed recall target, which method needs fewer candidates (= faster re-ranking)?  
Binary search is adaptive — its window always contains a fixed *count* of points.  
Hashing is non-adaptive — its window contains a fixed *range* in projection space.

In [ ]:
# Best bucket_size from the sweep above (use whichever gave highest R@K with fewest cands)
BEST_BSZ = 0.3   # update this after running the sweep
idx_hash_best = AMPIHashing(X, num_projections=16, bucket_size=BEST_BSZ, seed=SEED)

print(f'  Binary vs Hashing (w={BEST_BSZ}) — candidates vs recall@{K}')
print(f'  {"Method":<24}  {"cands":>8}  {"ms/q":>8}  {f"R@{K}":>6}')
print('  ' + '-' * 56)

rows_bin  = []
rows_hash = []

for w in [10, 25, 50, 100, 200, 400, 800]:
    fn = lambda q, w=w: idx_bin.query(q, k=K, window_size=w)[2]
    cf = lambda w=w: int(np.mean([len(idx_bin.query_candidates(q, window_size=w)) for q in queries[:20]]))
    rows  = [fn(q) for q in queries]
    I     = np.array([np.pad(r, (0, max(0, K - len(r))), constant_values=-1) for r in rows])
    ms    = warmup_time(fn, queries)
    cands = cf()
    rk    = recall_at_k(I_true, I, K)
    rows_bin.append((cands, rk))
    print(f'  {f"Binary w={w}":<24}  {cands:>8,}  {ms:>8.2f}  {rk:>6.3f}')

print()
for p in [1, 2, 3, 5, 10]:
    fn = lambda q, p=p: idx_hash_best.query(q, k=K, probes=p)[2]
    cf = lambda p=p: int(np.mean([len(idx_hash_best.query_candidates(q, probes=p)) for q in queries[:20]]))
    rows  = [fn(q) for q in queries]
    I     = np.array([np.pad(r, (0, max(0, K - len(r))), constant_values=-1) for r in rows])
    ms    = warmup_time(fn, queries)
    cands = cf()
    rk    = recall_at_k(I_true, I, K)
    rows_hash.append((cands, rk))
    print(f'  {f"Hash w={BEST_BSZ} p={p}":<24}  {cands:>8,}  {ms:>8.2f}  {rk:>6.3f}')

In [ ]:
# ── Empirical NN distance stats (helps calibrate bucket_size) ────────────────
# For each query, compute the distance to its 10th nearest neighbour.

nn_dists = np.sqrt(np.array([
    flat.search(q[None], K)[0][0, K-1] for q in queries[:200]
]))

sigma_proj = nn_dists / np.sqrt(d)

print(f'MNIST nearest-neighbour distances (L2, k={K})')
print(f'  r_nn  :  mean={nn_dists.mean():.2f}  median={np.median(nn_dists):.2f}  p10={np.percentile(nn_dists,10):.2f}  p90={np.percentile(nn_dists,90):.2f}')
print(f'  σ_proj = r_nn/√d:  mean={sigma_proj.mean():.3f}  median={np.median(sigma_proj):.3f}')
print(f'  Recommended bucket_size ≈ {np.median(sigma_proj):.2f}')